In [ ]:
import numpy as np

# Step 1: Real-world data representation (Web Graph of 5 pages)
# Let's define the links between 5 pages (0 to 4):
# Page 0 links to: 1, 2
# Page 1 links to: 2
# Page 2 links to: 0, 3, 4
# Page 3 links to: 4
# Page 4 links to: 0

num_pages = 5
adjacency_matrix = np.array([
    [0, 1, 1, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 1, 1],
    [0, 0, 0, 0, 1],
    [1, 0, 0, 0, 0]
], dtype=float)

# Step 2: Transition Matrix Construction (Column-stochastic matrix)
# Each column represents outgoing links from a page. Let's build the transition matrix M.
M = np.zeros((num_pages, num_pages))
for j in range(num_pages):
    num_outlinks = np.sum(adjacency_matrix[j])
    if num_outlinks > 0:
        M[:, j] = adjacency_matrix[j] / num_outlinks
    else:
        # Handling dead-ends (dangling nodes) by distributing probability equally
        M[:, j] = 1.0 / num_pages

print("Adjacency Transition Matrix (M):\n", M)

# Google's PageRank Formula: G = d * M + (1 - d) / N * J
# where d is the damping factor (typically 0.85), and J is an all-ones matrix
d = 0.85
teleport = np.ones((num_pages, num_pages)) / num_pages
G = d * M + (1 - d) * teleport
print("\nGoogle Matrix (G) with damping factor 0.85:\n", G)

# Step 3: Pattern discovery & System simplification (Eigenvalues & Eigenvectors)
# By Perron-Frobenius theorem, G has a unique largest eigenvalue of lambda = 1.
# The corresponding eigenvector represents the steady-state PageRank vector.

# Method A: Finding dominant eigenvector via numpy's linalg.eig
eigenvalues, eigenvectors = np.linalg.eig(G)

# Find the index of the eigenvalue closest to 1.0
idx = np.argmin(np.abs(eigenvalues - 1.0))
ranking_vector_eig = np.real(eigenvectors[:, idx])

# Normalize the eigenvector so that its elements sum to 1
ranking_vector_eig = ranking_vector_eig / np.sum(ranking_vector_eig)

# Method B: Power Iteration Method (Modeling/Simulation)
# Start with a uniform probability distribution
v = np.ones(num_pages) / num_pages
tolerance = 1e-08
max_iterations = 100

for i in range(max_iterations):
    v_next = G @ v
    if np.linalg.norm(v_next - v, 1) < tolerance:
        v = v_next
        print(f"\nPower iteration converged in {i+1} iterations.")
        break
    v = v_next

# Display Results
print("\nFinal PageRank Scores:")
for page_id, score in enumerate(v):
    print(f"  Page {page_id}: {score:.5f} (Eigenvector method: {ranking_vector_eig[page_id]:.5f})")

# Rank the pages
ranked_indices = np.argsort(v)[::-1]
print("\nPage Ranking order (Highest to Lowest):")
for rank, page_id in enumerate(ranked_indices):
    print(f"  Rank {rank+1}: Page {page_id} with score {v[page_id]:.5f}")

Adjacency Transition Matrix (M):
 [[0.         0.         0.33333333 0.         1.        ]
 [0.5        0.         0.         0.         0.        ]
 [0.5        1.         0.         0.         0.        ]
 [0.         0.         0.33333333 0.         0.        ]
 [0.         0.         0.33333333 1.         0.        ]]

Google Matrix (G) with damping factor 0.85:
 [[0.03       0.03       0.31333333 0.03       0.88      ]
 [0.455      0.03       0.03       0.03       0.03      ]
 [0.455      0.88       0.03       0.03       0.03      ]
 [0.03       0.03       0.31333333 0.03       0.03      ]
 [0.03       0.03       0.31333333 0.88       0.03      ]]

Power iteration converged in 35 iterations.

Final PageRank Scores:
  Page 0: 0.27552 (Eigenvector method: 0.27552)
  Page 1: 0.14710 (Eigenvector method: 0.14710)
  Page 2: 0.27213 (Eigenvector method: 0.27213)
  Page 3: 0.10710 (Eigenvector method: 0.10710)
  Page 4: 0.19814 (Eigenvector method: 0.19814)

Page Ranking order (Highest 